# Compare Experiments
Comprehensive cross-run view for training, evaluation, and calibration. Adjust selection in the cells below.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Plot theming
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams.update({'figure.dpi': 300, 'savefig.dpi': 300, 'axes.titlesize': 12, 'axes.labelsize': 11, 'legend.fontsize': 10})

PLOTS_DIR = Path('../comparison_plots')
PLOTS_DIR.mkdir(exist_ok=True)

def save_fig(fig, name: str, prefix: str = 'comparison'):
    path = PLOTS_DIR / f"{prefix}_{name}.pdf"
    fig.savefig(path, bbox_inches='tight', format='pdf')
    print(f"Saved figure to {path}")
    return path


In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd

# Prefer local src/ over installed package
sys.path.insert(0, str((Path('..').resolve() / 'src')))

RUNS_ROOT = Path('../training_runs')
RUNS_ROOT.mkdir(exist_ok=True)

# Selection controls
SELECTED_EXPERIMENTS = None  # e.g., ['experiment_resnet18_20251206-104814']
EXCLUDE_PATTERNS = []       # e.g., ['debug']
SORT_BY = 'best_val_f1'
SORT_ASC = False
MAX_CURVE_PLOTS = 5  # limit number of runs to overlay for curves


In [ ]:
def load_json(path: Path):
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return {}

def should_include(path: Path) -> bool:
    return path.is_dir() and not any(pattern in path.name for pattern in EXCLUDE_PATTERNS)

exp_dirs = [p for p in RUNS_ROOT.iterdir() if should_include(p)]
if SELECTED_EXPERIMENTS:
    exp_dirs = [RUNS_ROOT / name for name in SELECTED_EXPERIMENTS]

records = []
histories = {}
for exp_dir in sorted(exp_dirs):
    cfg = load_json(exp_dir / 'config.json')
    train_summary = load_json(exp_dir / 'train_summary.json')
    test_metrics = load_json(exp_dir / 'test_metrics.json')
    eval_metrics = load_json(exp_dir / 'eval_metrics.json')
    history = train_summary.get('history', []) if isinstance(train_summary, dict) else []

    best_epoch = None
    best_val_f1 = None
    best_val_acc = None
    best_val_loss = None
    last_val_f1 = None
    last_val_acc = None
    last_val_loss = None
    train_loss_min = None
    epochs_ran = len(history)
    if history:
        best_idx = max(range(len(history)), key=lambda i: history[i].get('val/macro_f1', float('-inf')))
        best = history[best_idx]
        best_epoch = best.get('epoch')
        best_val_f1 = best.get('val/macro_f1')
        best_val_acc = best.get('val/accuracy')
        best_val_loss = best.get('val_loss')
        last = history[-1]
        last_val_f1 = last.get('val/macro_f1')
        last_val_acc = last.get('val/accuracy')
        last_val_loss = last.get('val_loss')
        train_loss_min = min((h.get('train_loss') for h in history if 'train_loss' in h), default=None)
        histories[exp_dir.name] = history

    eval_val = eval_metrics.get('val', {}) if isinstance(eval_metrics, dict) else {}
    eval_test = eval_metrics.get('test', {}) if isinstance(eval_metrics, dict) else {}

    overfit_gap = None
    if best_val_f1 is not None and eval_test.get('macro_f1') is not None:
        overfit_gap = best_val_f1 - eval_test.get('macro_f1')

    records.append({
        'experiment': exp_dir.name,
        'model': cfg.get('model_name'),
        'data_root': cfg.get('data_root'),
        'balance_strategy': cfg.get('balance_strategy'),
        'batch_size': cfg.get('batch_size'),
        'optimizer': cfg.get('optimizer'),
        'lr': cfg.get('lr'),
        'weight_decay': cfg.get('weight_decay'),
        'label_smoothing': cfg.get('label_smoothing'),
        'epochs_planned': cfg.get('epochs'),
        'epochs_ran': epochs_ran,
        'best_epoch': best_epoch,
        'best_val_f1': best_val_f1,
        'best_val_accuracy': best_val_acc,
        'best_val_loss': best_val_loss,
        'last_val_f1': last_val_f1,
        'last_val_accuracy': last_val_acc,
        'last_val_loss': last_val_loss,
        'train_loss_min': train_loss_min,
        'eval_val_macro_f1': eval_val.get('macro_f1'),
        'eval_val_accuracy': eval_val.get('accuracy'),
        'eval_val_brier': eval_val.get('brier'),
        'eval_val_ece': eval_val.get('ece'),
        'eval_test_macro_f1': eval_test.get('macro_f1'),
        'eval_test_accuracy': eval_test.get('accuracy'),
        'eval_test_brier': eval_test.get('brier'),
        'eval_test_ece': eval_test.get('ece'),
        'test_macro_f1': test_metrics.get('macro_f1') if isinstance(test_metrics, dict) else None,
        'test_accuracy': test_metrics.get('accuracy') if isinstance(test_metrics, dict) else None,
        'overfit_gap': overfit_gap,
    })

if not records:
    raise FileNotFoundError(f'No experiments found under {RUNS_ROOT}')

df = pd.DataFrame.from_records(records)
if SORT_BY in df.columns:
    df = df.sort_values(SORT_BY, ascending=SORT_ASC)

core_cols = [
    'experiment', 'model', 'data_root', 'balance_strategy', 'batch_size',
    'optimizer', 'lr', 'weight_decay', 'label_smoothing',
    'epochs_planned', 'epochs_ran', 'best_epoch',
    'best_val_f1', 'best_val_accuracy', 'best_val_loss',
    'last_val_f1', 'last_val_accuracy', 'last_val_loss',
    'eval_val_macro_f1', 'eval_val_accuracy', 'eval_val_brier', 'eval_val_ece',
    'eval_test_macro_f1', 'eval_test_accuracy', 'eval_test_brier', 'eval_test_ece',
    'test_macro_f1', 'test_accuracy',
    'overfit_gap',
]

print(f"Aggregated {len(df)} experiments")
display(df[core_cols])


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df.empty:
    print('No data to plot yet.')
else:
    # Macro F1 comparisons
    metric_cols = ['best_val_f1', 'eval_val_macro_f1', 'eval_test_macro_f1', 'test_macro_f1']
    melted = df.melt(id_vars=['experiment', 'model'], value_vars=[c for c in metric_cols if c in df.columns])
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=melted, x='experiment', y='value', hue='variable', ax=ax)
    ax.set_ylabel('Macro F1')
    ax.set_xlabel('Experiment')
    ax.set_title('F1 Comparison')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    save_fig(fig, 'macro_f1')
    plt.show()

    # Calibration (ECE)
    calib_cols = ['eval_val_ece', 'eval_test_ece']
    calib_melt = df.melt(id_vars=['experiment', 'model'], value_vars=[c for c in calib_cols if c in df.columns])
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.barplot(data=calib_melt, x='experiment', y='value', hue='variable', ax=ax)
    ax.set_ylabel('ECE')
    ax.set_xlabel('Experiment')
    ax.set_title('Expected Calibration Error (lower is better)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    save_fig(fig, 'ece')
    plt.show()

    # Brier scores
    brier_cols = ['eval_val_brier', 'eval_test_brier']
    brier_melt = df.melt(id_vars=['experiment', 'model'], value_vars=[c for c in brier_cols if c in df.columns])
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.barplot(data=brier_melt, x='experiment', y='value', hue='variable', ax=ax)
    ax.set_ylabel('Brier score (lower is better)')
    ax.set_xlabel('Experiment')
    ax.set_title('Brier Score Comparison')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    save_fig(fig, 'brier')
    plt.show()

    # Overfitting gap
    if 'overfit_gap' in df.columns:
        fig, ax = plt.subplots(figsize=(12, 4))
        sns.barplot(data=df, x='experiment', y='overfit_gap', hue='model', ax=ax)
        ax.axhline(0, color='gray', linestyle='--')
        ax.set_ylabel('Best Val F1 - Eval Test F1')
        ax.set_xlabel('Experiment')
        ax.set_title('Generalization gap (negative/low is better)')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        save_fig(fig, 'overfit_gap')
        plt.show()

    # Scatter: best val vs test F1
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(df['best_val_f1'], df['eval_test_macro_f1'], c='tab:blue')
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray')
    for _, row in df.iterrows():
        ax.text(row['best_val_f1'] + 0.005, row['eval_test_macro_f1'] + 0.005, row['experiment'], fontsize=8)
    ax.set_xlabel('Best Val F1 (train history)')
    ax.set_ylabel('Test F1 (eval)')
    ax.set_title('Val vs Test F1')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    save_fig(fig, 'val_vs_test_f1')
    plt.show()

    # Optional: training curves for top runs
    if histories:
        # pick top-N by best_val_f1
        top_exp = df.sort_values('best_val_f1', ascending=False).head(MAX_CURVE_PLOTS)['experiment']
        fig, ax = plt.subplots(figsize=(10, 5))
        for exp in top_exp:
            hist = histories.get(exp, [])
            if not hist:
                continue
            epochs = [h.get('epoch') for h in hist]
            vals = [h.get('val/macro_f1') for h in hist]
            ax.plot(epochs, vals, marker='o', label=exp)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Val Macro F1')
        ax.set_title('Validation F1 per Epoch (top runs)')
        ax.legend()
        plt.tight_layout()
        save_fig(fig, 'val_f1_curves')
        plt.show()

        fig, ax = plt.subplots(figsize=(10, 5))
        for exp in top_exp:
            hist = histories.get(exp, [])
            if not hist:
                continue
            epochs = [h.get('epoch') for h in hist]
            losses = [h.get('val_loss') for h in hist]
            ax.plot(epochs, losses, marker='o', label=exp)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Val Loss')
        ax.set_title('Validation Loss per Epoch (top runs)')
        ax.legend()
        plt.tight_layout()
        save_fig(fig, 'val_loss_curves')
        plt.show()
